# Introduction

Since Jan. 1, 2015, [The Washington Post](https://www.washingtonpost.com/) has been compiling a database of every fatal shooting in the US by a police officer in the line of duty. 

<center><img src=https://i.imgur.com/sX3K62b.png></center>

While there are many challenges regarding data collection and reporting, The Washington Post has been tracking more than a dozen details about each killing. This includes the race, age and gender of the deceased, whether the person was armed, and whether the victim was experiencing a mental-health crisis. The Washington Post has gathered this supplemental information from law enforcement websites, local news reports, social media, and by monitoring independent databases such as "Killed by police" and "Fatal Encounters". The Post has also conducted additional reporting in many cases.

There are 4 additional datasets: US census data on poverty rate, high school graduation rate, median household income, and racial demographics. [Source of census data](https://factfinder.census.gov/faces/nav/jsf/pages/community_facts.xhtml).

### Upgrade Plotly

Run the cell below if you are working with Google Colab

In [1]:
%pip install --upgrade plotly pip

Note: you may need to restart the kernel to use updated packages.


## Import Statements

In [2]:
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns

# This might be helpful:
from collections import Counter

## Notebook Presentation

In [3]:
pd.options.display.float_format = '{:,.2f}'.format

## Load the Data

In [4]:
df_hh_income = pd.read_csv('data/Median_Household_Income_2015.csv', encoding="windows-1252")
df_pct_poverty = pd.read_csv('data/Pct_People_Below_Poverty_Level.csv', encoding="windows-1252")
df_pct_completed_hs = pd.read_csv('data/Pct_Over_25_Completed_High_School.csv', encoding="windows-1252")
df_share_race_city = pd.read_csv('data/Share_of_Race_By_City.csv', encoding="windows-1252")
df_fatalities = pd.read_csv('data/Deaths_by_Police_US.csv', encoding="windows-1252")

# Preliminary Data Exploration

* What is the shape of the DataFrames? 
* How many rows and columns do they have?
* What are the column names?
* Are there any NaN values or duplicates?

In [5]:
print(df_hh_income.shape)
print(df_hh_income.columns.tolist())
print(df_hh_income.head())

(29322, 3)
['Geographic Area', 'City', 'Median Income']
  Geographic Area             City Median Income
0              AL       Abanda CDP         11207
1              AL   Abbeville city         25615
2              AL  Adamsville city         42575
3              AL     Addison town         37083
4              AL       Akron town         21667


In [6]:
print(df_pct_poverty.shape)
print(df_pct_poverty.columns.tolist())
print(df_pct_poverty.head())

(29329, 3)
['Geographic Area', 'City', 'poverty_rate']
  Geographic Area             City poverty_rate
0              AL       Abanda CDP         78.8
1              AL   Abbeville city         29.1
2              AL  Adamsville city         25.5
3              AL     Addison town         30.7
4              AL       Akron town           42


In [7]:
print(df_pct_completed_hs.shape)
print(df_pct_completed_hs.columns.tolist())
print(df_pct_completed_hs.head())

(29329, 3)
['Geographic Area', 'City', 'percent_completed_hs']
  Geographic Area             City percent_completed_hs
0              AL       Abanda CDP                 21.2
1              AL   Abbeville city                 69.1
2              AL  Adamsville city                 78.9
3              AL     Addison town                 81.4
4              AL       Akron town                 68.6


In [8]:
print(df_share_race_city.shape)
print(df_share_race_city.columns.tolist())
print(df_share_race_city.head())

(29268, 7)
['Geographic area', 'City', 'share_white', 'share_black', 'share_native_american', 'share_asian', 'share_hispanic']
  Geographic area             City share_white share_black  \
0              AL       Abanda CDP        67.2        30.2   
1              AL   Abbeville city        54.4        41.4   
2              AL  Adamsville city        52.3        44.9   
3              AL     Addison town        99.1         0.1   
4              AL       Akron town        13.2        86.5   

  share_native_american share_asian share_hispanic  
0                     0           0            1.6  
1                   0.1           1            3.1  
2                   0.5         0.3            2.3  
3                     0         0.1            0.4  
4                     0           0            0.3  


In [9]:
print(df_fatalities.shape)
print(df_fatalities.columns.tolist())
print(df_fatalities.head())

(2535, 14)
['id', 'name', 'date', 'manner_of_death', 'armed', 'age', 'gender', 'race', 'city', 'state', 'signs_of_mental_illness', 'threat_level', 'flee', 'body_camera']
   id                name      date   manner_of_death       armed   age  \
0   3          Tim Elliot  02/01/15              shot         gun 53.00   
1   4    Lewis Lee Lembke  02/01/15              shot         gun 47.00   
2   5  John Paul Quintero  03/01/15  shot and Tasered     unarmed 23.00   
3   8     Matthew Hoffman  04/01/15              shot  toy weapon 32.00   
4   9   Michael Rodriguez  04/01/15              shot    nail gun 39.00   

  gender race           city state  signs_of_mental_illness threat_level  \
0      M    A        Shelton    WA                     True       attack   
1      M    W          Aloha    OR                    False       attack   
2      M    H        Wichita    KS                    False        other   
3      M    W  San Francisco    CA                     True       attack   

## Data Cleaning - Check for Missing Values and Duplicates

Dealing with the NaN values. Perhaps substituting 0 is appropriate.

In [10]:
for name, df in [("hh_income", df_hh_income), ("pct_poverty", df_pct_poverty),
                 ("pct_completed_hs", df_pct_completed_hs), ("share_race_city", df_share_race_city),
                 ("fatalities", df_fatalities)]:
    print(f"\n--- {name} ---")
    print("NaNs:\n", df.isna().sum())
    print("Duplicates:", df.duplicated().sum())


--- hh_income ---
NaNs:
 Geographic Area     0
City                0
Median Income      51
dtype: int64
Duplicates: 0

--- pct_poverty ---
NaNs:
 Geographic Area    0
City               0
poverty_rate       0
dtype: int64
Duplicates: 0

--- pct_completed_hs ---
NaNs:
 Geographic Area         0
City                    0
percent_completed_hs    0
dtype: int64
Duplicates: 0

--- share_race_city ---
NaNs:
 Geographic area          0
City                     0
share_white              0
share_black              0
share_native_american    0
share_asian              0
share_hispanic           0
dtype: int64
Duplicates: 0

--- fatalities ---
NaNs:
 id                           0
name                         0
date                         0
manner_of_death              0
armed                        9
age                         77
gender                       0
race                       195
city                         0
state                        0
signs_of_mental_illness      0
threat_le

In [11]:
print(df_fatalities["race"].unique())

['A' 'W' 'H' 'B' 'O' nan 'N']


In [12]:
# Fix inconsistent colmn name
df_share_race_city.rename(columns={'Geographic area': 'Geographic Area'}, inplace=True)

# Fill NaNs - modern pandas style
df_hh_income["Median Income"] = df_hh_income["Median Income"].fillna(0)
df_fatalities["armed"] = df_fatalities["armed"].fillna("Unknown")
df_fatalities["race"] = df_fatalities["race"].fillna("Unknown")
df_fatalities["flee"] = df_fatalities["flee"].fillna("Unknown")

# Fix types
df_fatalities["age"] = df_fatalities["age"].astype("Int64")
df_fatalities["date"] = pd.to_datetime(df_fatalities["date"], format="%d/%m/%y")

# Map race abbreviations
race_map = {"A": "Asian", "W": "White", "H": "Hispanic", "B": "Black", "N": "Native American", "O": "Other", "Unknown": "Unknown"}
df_fatalities["race"] = df_fatalities["race"].map(race_map)

print(df_fatalities.dtypes)
print(df_fatalities.head())

id                                  int64
name                               object
date                       datetime64[ns]
manner_of_death                    object
armed                              object
age                                 Int64
gender                             object
race                               object
city                               object
state                              object
signs_of_mental_illness              bool
threat_level                       object
flee                               object
body_camera                          bool
dtype: object
   id                name       date   manner_of_death       armed  age  \
0   3          Tim Elliot 2015-01-02              shot         gun   53   
1   4    Lewis Lee Lembke 2015-01-02              shot         gun   47   
2   5  John Paul Quintero 2015-01-03  shot and Tasered     unarmed   23   
3   8     Matthew Hoffman 2015-01-04              shot  toy weapon   32   
4   9   Michael Rodrigu

# Poverty Rate in each US State

A bar chart that ranks the poverty rate from highest to lowest by US state. Which state has the highest poverty rate? Which state has the lowest poverty rate? Bar Plot

# High School Graduation Rate by US State

High School Graduation Rate in ascending order of US States. Which state has the lowest high school graduation rate? Which state has the highest?

# Visualize the Relationship between Poverty Rates and High School Graduation Rates

#### A line chart with two y-axes to show if the rations of poverty and high school graduation move together.

#### Seaborn .jointplot() with a Kernel Density Estimate (KDE) and/or scatter plot to visualize the same relationship

#### Seaborn's `.lmplot()` or `.regplot()` to show a linear regression between the poverty ratio and the high school graduation ratio. 

# Bar Chart with Subsections Showing the Racial Makeup of Each US State

Visualize the share of the white, black, hispanic, asian and native american population in each US State using a bar chart with sub sections.

# Donut Chart by of People Killed by Race

Use `.value_counts()`

# Chart Comparing the Total Number of Deaths of Men and Women

`df_fatalities` illustrates how many more men are killed compared to women.

# Box Plot Showing the Age and Manner of Death

Break out the data by gender using `df_fatalities`. Is there a difference between men and women in the manner of death? 

# Were People Armed? 

In what percentage of police killings were people armed? A chart showing what kind of weapon (if any) the deceased was carrying. How many of the people killed by police were armed with guns versus unarmed?

# How Old Were the People Killed?

Percentage of people killed were under 25 years old.

Histogram and KDE plot that shows the distribution of ages of the people killed by police.

Separate KDE plot for each race. Is there a difference between the distributions?

# Race of People Killed

Chart that shows the total number of people killed by race.

# Mental Illness and Police Killings

Percentage of people killed by police have been diagnosed with a mental illness.

# Cities where the Most Police Killings Take Place?

Chart ranking the top 10 cities with the most police killings. Which cities are the most dangerous?

# Rate of Death by Race

The share of each race in the top 10 cities. Contrast this with the top 10 cities of police killings to work out the rate at which people are killed by race for each city.

# Choropleth Map of Police Killings by US State

Which states are the most dangerous? Compare map with previous chart. Are these the same states with high degrees of poverty?

# Number of Police Killings Over Time

Number of Police Killings over Time. Is there a trend in the data?

# Epilogue

Now that the analysis of the data is done, read [The Washington Post's analysis here](https://www.washingtonpost.com/graphics/investigations/police-shootings-database/).